Mi respuesta a preguntas del tp que estan en el medio del codigo

- Q3) Es para calcular la frecuencia absoluta de cada label
- Q4) Podria, pero la implementacion de _fit_params usa a_priori, asi que se deberia reescribir parte del codigo
- Q5) El masking debe ser un vector no una matriz de una columna/fila
- Q6) Pq nuestra implementacion utiliza el estimador de max likelihood
- Q7) Pq queremos la media muestral de cada feature para cada label, no la media de la observacion para cada label

___

In [1]:
# imports
import numpy as np
import numpy.linalg as LA

from base.qda import QDA, TensorizedQDA
from base.cholesky import QDA_Chol1, QDA_Chol2, QDA_Chol3
from utils.bench import Benchmark
from utils.datasets import (get_iris_dataset, get_letters_dataset, 
                            get_penguins_dataset, get_wine_dataset,
                            label_encode)

1. ¿Sobre qué paraleliza `TensorizedQDA`? ¿Sobre las $k$ clases, las $n$ observaciones a predecir, o ambas?

Paraleliza sobre las clases, ya que evita el for loop antes usado en predict_one. Aun asi, el metodo predict no se modifica y este tanto en QDA como en TensorizedQDA usa un for loop para iterar en cada sample

___

2. Analizar los shapes de `tensor_inv_covs` y `tensor_means` y explicar paso a paso cómo es que `TensorizedQDA` llega a predecir lo mismo que `QDA`.

In [2]:
X_full, y_full = get_wine_dataset()
y_full_encoded = label_encode(y_full)
print(f"Shape de X -> {X_full.shape} | Shape de Y -> {y_full_encoded.shape}")

Shape de X -> (178, 13) | Shape de Y -> (178, 1)


In [3]:
X_full, y_full = get_wine_dataset()

tqda = TensorizedQDA()
tqda.fit(X_full.T,y_full_encoded.T)

print(f"Shape de tensor_inv_covs -> {tqda.tensor_inv_cov.shape}")
print(f"Shape de tensor_means -> {tqda.tensor_means.shape}")

Shape de tensor_inv_covs -> (3, 13, 13)
Shape de tensor_means -> (3, 13, 1)


Los shapes de estos tensores se pueden entender como que el primer indice selecciona una clase en parrticular. En el caso de la matriz inversa de covarianza el segundo y tercer indice hacen referencia a los elementos de la matriz de covarianza. Por otro lado, el de las medias se puede entender como una matriz de una columna donde cada fila reprresenta la media de cada feature.

Llegan a la misma solucion ya que hacen un camino similar parteindo del metodo predict, con la diferencia en la implementacion de `predict_one`. En el de QDA en un for loop hace la estimacion iterando en cada clase, llamando de uno a uno al metodo `_predict_log_conditional`. Por otro lado, en TensorizedQDA se reemplimenta `predict_one` haciendo un unico llamado a `_predict_log_conditional` en donde el calculo se hace con los tensores. En este caso, numpy al hacer operaciones entre matrices y tensores la replica en las dimensiones faltantes. Por ej cuando se hace `x - self.tensor_means`, donde x es de Nx1 y self_tensor_means es de KxNx1  se replica la resta de las matrices Nx1 en las K dimensiones. Algo similar pasa con la otra operacion, en donde se ajusta la trnaspuesta para que tome efecto en las matrices asociadas a cada label y se calcula de forma similar, con la precaucion de que ahora el determinate calculado tiene forma de un vector por lo que se ajusta el producto matricial para que las dimensiones coincidan.

___


3. Implementar el modelo `FasterQDA` (se recomienda heredarlo de `TensorizedQDA`) de manera de eliminar el ciclo for en el método predict.

In [45]:
class FasterQDA(TensorizedQDA):
    def predict(self, X):

        unbiased_x = X - self.tensor_means
        inner_prod = unbiased_x.transpose(0,2,1) @ self.tensor_inv_cov @ unbiased_x
        diag_inner_prod = np.diagonal(inner_prod,axis1=1,axis2=2).T # Mantenemos solo la diagonal, transpongo para poder hacer los calculos
        log_cond = 0.5*np.log(LA.det(self.tensor_inv_cov)) - 0.5 * diag_inner_prod
        posteriori = self.log_a_priori + log_cond

        y_hat = np.argmax(posteriori,axis=1) # Probabilidades en columnas
    
        return y_hat.reshape(1,-1)


In [ ]:
# Check rapido de implementacion

fqda = FasterQDA()
fqda.fit(X_full.T,y_full_encoded.T)

a = tqda.predict(X_full.T)
b = fqda.predict(X_full.T)

print(f"Son iguales: {(a==b).all()}")


Son iguales: True


___

4. Mostrar dónde aparece la mencionada matriz de $n \times n$, donde $n$ es la cantidad de observaciones a predecir.

Esta matriz aparece al hacer $$(X-\mu)^T \Sigma^{-1} (X- \mu)$$

En la implementacion de FasterQDA se observa en la variable `inner_prod`

___

5. Demostrar que
$$
diag(A \cdot B) = \sum_{cols} A \odot B^T = np.sum(A \odot B^T, axis=1)
$$ es decir, que se puede "esquivar" la matriz de $n \times n$ usando matrices de $n \times p$. También se puede usar, de forma equivalente,
$$
np.sum(A^T \odot B, axis=0).T
$$